# Introduction to `RandomForest`

# Importing the data
These files contain previously prepared training/test features. We will compare classifiers on this supplied representation; the original raw training data and fitted preprocessor are not supplied here.

Consequently, CV below evaluates **conditional on the prepared representation**. It cannot independently validate how upstream preprocessing was fitted. For a fully independent end-to-end estimate, obtain the raw data and put learned preprocessing inside the CV pipeline. Keep the supplied test set untouched while comparing and tuning models.

In [ ]:
# Reading the files
import pandas as pd

X_train = pd.read_csv('../../Excercise_data/L06/X_train_prep.csv')
X_test = pd.read_csv('../../Excercise_data/L06/X_test_prep.csv')

y_train = pd.read_csv('../../Excercise_data/L06/y_train.csv').values.ravel() # ravel returns a one-dimensional target array
y_test = pd.read_csv('../../Excercise_data/L06/y_test.csv').values.ravel()

### Creating a smaller training set
The supplied training set contains 225,605 observations. For a manageable classroom comparison, draw a stratified subset of 5,000 observations, then split it into fitting and validation subsets. Reserve the full training set for refitting the final selected model.

The fixed seed makes the comparison reproducible. Smaller samples speed the search but may change which settings look best.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_small, _, y_train_small, _ = train_test_split(
    X_train, y_train, train_size=5000, stratify=y_train, random_state=42)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_small, y_train_small, test_size=0.2,
    stratify=y_train_small, random_state=42)

# GridSearch different algorithms
Create a `Pipeline` with a step named `classifier`, then compare:
- `SVC()`
- `DecisionTreeClassifier(random_state=42)`
- `RandomForestClassifier(random_state=42)`
- `LogisticRegression(max_iter=3000)`
- `KNeighborsClassifier()`

Use `GridSearchCV` with `cv=3` and `scoring='precision'` on `X_fit, y_fit`. Save `best_params_` as `best_parameters`. This compact classroom search compares the algorithms on the same observations.

If a candidate predicts no positive examples, its precision is undefined. Scikit-learn reports 0 and issues a warning under this scorer. Inspect the prediction counts; this is an informative failure case, not evidence that precision alone is sufficient.

In [ ]:
#

# A critical look
Evaluate the selected model on `X_val, y_val`. Save its precision and recall as `precision_score_rf` and `recall_score_rf`. Does selecting for precision alone meet both aims? Keep the test set closed while deciding how to tune.

In [ ]:
#

<details><summary>Conclusions</summary>
High precision can coexist with low recall. F1 is the harmonic mean of precision and recall; increasing it does **not** guarantee that both increase separately. Inspect both and choose the metric that matches the application before the next search.
</details>

## Narrowing our Hyperparameter Space with RandomizedSearchCV
Now focus on Random Forest as an ensemble example, whether or not it won the initial comparison. Use `scoring='f1'` consistently and search these representative values:
- `classifier__max_depth`: 5, 10, 20, None
- `classifier__n_estimators`: 50, 100
- `classifier__min_samples_split`: 2, 5, 10
- `classifier__max_features`: 'sqrt', 0.5, 1.0

Use `X_fit, y_fit` with `cv=3`, `n_iter=6` and `random_state=42`. Six candidates keep the classroom run short; expanding the data/search is optional. Set the forest's `random_state=42` too.

Save `best_score_` as `best_score` and `best_estimator_` as `rf_model`. Compare validation precision (`rf_precision`), recall (`rf_recall`) and F1 with a default forest fitted on the same fitting set. Decide whether to retain the tuned or default model using validation results, then refit that choice on the **full training set** and evaluate the test set once.

In [ ]:
#

### Why Ensemble Learning Can Work
Random Forest averages the class probabilities from trees fitted on bootstrap samples, with random candidate features at each split. Aggregation can reduce variance when useful trees make imperfectly correlated errors. Errors are not guaranteed to be Gaussian or centred on the correct answer.

First examine a different idea: **cross-validation variability**. Use `DecisionTreeClassifier(random_state=42)` directly on the prepared `X_train_small, y_train_small`; there is no supplied `preproc` object. Run `cross_validate` with `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` and `scoring='accuracy'`, and plot the five `test_score` values. Use the same folds below.

In [ ]:
#

<details><summary>🌳 Observations</summary>
The fold scores vary because the training and held-out observations differ. Five scores do not establish a Gaussian error distribution. A high held-out score alone is not evidence of overfitting; inspect training/validation behaviour and sampling variability.
</details>

## Out-of-fold predictions are not an ensemble
Use `cross_val_predict` with the same decision tree, five folds and `X_train_small, y_train_small`. Save predictions as `cv_predict`.

Each row receives **one** prediction from the model fitted without its fold. This is not an average of several models' predictions for that row. Calculate `accuracy_score(y_train_small, cv_predict)` and save it as `cv_score`.

Compare it with a Random Forest evaluated with the **same** out-of-fold procedure. This comparison changes the fitted estimator, not the evaluation rows.

In [ ]:
#

<details><summary>🌳 Observations</summary>
A Random Forest combines many trees within each fold; `cross_val_predict` then supplies one held-out forest prediction per row. Random feature subsets can reduce correlation between trees, but greater tree variability alone does not guarantee better predictions. Compare the observed scores rather than assuming a winner.
</details>

## Feature Importance
Inspect `rf_model.named_steps['classifier'].feature_importances_` and make a sorted `pd.Series` with `X_train.columns` as the index.

These are normalised, sample-weighted impurity decreases accumulated over tree splits and averaged over the forest. They can favour high-cardinality features and distribute importance unpredictably among correlated predictors. A small value does not establish that a feature is intrinsically irrelevant; a large value is not a causal effect.

As a check, compute permutation importance on validation data, without using the test set to choose features.

In [ ]:
#

## Saving Time with Bayesian Optimization
When tuning complex models (like RandomForest), brute-force methods such as Grid Search or Random Search can be inefficient and time-consuming.

I believe you know by now and are still waiting for your Gridsearch to run while reading this!  😅



Bayesian Optimization, however, uses a probabilistic model (usually a Gaussian Process) to approximate the objective function and guide the search, making better choices for the next set of hyperparameters to evaluate.

